# ecoli3: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('13_ecoli3_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,75.083,62.080,66.944,60.000,96.333,78.167,72.707,95.524,68.582
smote,86.352,64.105,54.061,82.857,91.197,87.027,66.466,94.936,75.228
bsmote,84.812,60.115,49.111,81.429,89.530,85.479,66.439,94.746,72.464
ros,86.439,62.263,51.177,84.286,90.038,87.162,69.408,95.127,74.351
adasyn,86.011,60.672,49.185,84.286,89.208,86.747,65.884,94.511,73.342
casb,85.754,61.490,50.722,82.857,90.041,86.449,69.361,95.271,73.622
casb_relaxed,86.337,64.153,54.248,82.857,91.194,87.026,69.085,95.481,75.245
casb_edited,85.868,59.739,47.676,84.286,88.541,86.413,69.797,95.002,72.804
casb_ros,87.757,62.774,50.522,87.143,89.710,88.427,69.121,95.343,75.265


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,73.042,61.048,74.286,57.143,96.667,76.905,66.390,91.263,67.045
smote,84.518,59.051,48.244,81.429,88.877,85.153,55.960,90.364,71.785
bsmote,85.972,59.667,47.190,84.286,88.369,86.327,54.399,90.042,72.819
ros,81.915,54.837,43.146,77.143,87.874,82.509,55.367,89.588,68.376
adasyn,85.366,57.339,44.134,84.286,87.208,85.747,57.870,90.292,71.353
casb,83.666,59.455,49.193,78.571,89.874,84.223,57.445,90.588,71.560
casb_relaxed,85.882,61.873,51.025,82.857,90.038,86.448,61.613,91.132,73.878
casb_edited,84.700,56.965,44.148,82.857,87.380,85.118,56.893,91.128,70.833
casb_ros,83.363,58.030,46.958,78.571,89.205,83.888,56.480,90.349,70.696


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,68.603,58.278,75.119,50.000,98.000,74.000,69.675,93.426,63.440
smote,78.942,59.668,55.237,68.571,93.352,80.962,59.620,93.162,69.305
bsmote,80.539,62.931,60.346,70.000,93.858,81.929,60.139,93.212,71.735
ros,77.717,59.442,57.440,65.714,93.855,79.785,62.780,92.304,68.579
adasyn,78.930,57.430,51.335,70.000,92.036,81.018,59.979,92.917,68.180
casb,75.944,58.727,57.601,62.857,94.514,78.685,63.348,93.407,67.336
casb_relaxed,77.401,58.923,57.190,65.714,93.852,79.783,61.322,92.910,68.162
casb_edited,86.350,62.921,51.538,82.857,90.527,86.692,63.628,93.386,74.636
casb_ros,75.921,59.411,59.275,62.857,94.844,78.851,63.210,93.901,67.666


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,67.407,55.610,68.952,48.571,97.339,72.955,65.359,92.814,61.508
smote,88.084,67.070,57.186,85.714,92.197,88.956,65.147,92.506,77.577
bsmote,89.354,66.841,55.288,88.571,91.033,89.802,64.801,92.580,78.097
ros,88.117,66.612,56.500,85.714,91.697,88.706,65.389,92.196,77.365
adasyn,89.195,66.091,54.209,88.571,90.702,89.637,64.167,92.007,77.643
casb,87.388,66.855,57.548,84.286,92.527,88.407,66.123,92.577,77.121
casb_relaxed,87.557,67.247,57.896,84.286,92.525,88.405,66.213,92.743,77.402
casb_edited,88.837,64.970,53.111,88.571,90.038,89.305,65.673,92.696,76.904
casb_ros,88.384,68.008,58.214,85.714,92.527,89.121,66.130,92.578,78.196


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,71.033,59.254,71.325,53.929,97.085,75.507,68.533,93.257,65.144
smote,84.474,62.473,53.682,79.643,91.406,85.524,61.798,92.742,73.474
bsmote,85.169,62.388,52.984,81.071,90.697,85.884,61.445,92.645,73.779
ros,83.547,60.788,52.066,78.214,90.866,84.540,63.236,92.304,72.168
adasyn,84.875,60.383,49.716,81.786,89.788,85.787,61.975,92.432,72.629
casb,83.188,61.632,53.766,77.143,91.739,84.441,64.069,92.961,72.410
casb_relaxed,84.294,63.049,55.090,78.929,91.902,85.415,64.558,93.066,73.672
casb_edited,86.439,61.149,49.118,84.643,89.122,86.882,63.998,93.053,73.794
casb_ros,83.856,62.056,53.742,78.571,91.572,85.072,63.735,93.043,72.956


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_ros,87.757,ros,86.439,1.318,WIN
1,f1,casb_relaxed,64.153,smote,64.105,0.048,WIN
2,precision,casb_relaxed,54.248,smote,54.061,0.187,WIN
3,recall,casb_ros,87.143,ros,84.286,2.857,WIN


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,85.882,bsmote,85.972,-0.090,LOSS
1,f1,casb_relaxed,61.873,bsmote,59.667,2.206,WIN
2,precision,casb_relaxed,51.025,smote,48.244,2.781,WIN
3,recall,casb_relaxed,82.857,bsmote,84.286,-1.429,LOSS


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,86.350,bsmote,80.539,5.811,WIN
1,f1,casb_edited,62.921,bsmote,62.931,-0.009,LOSS
2,precision,casb_ros,59.275,bsmote,60.346,-1.071,LOSS
3,recall,casb_edited,82.857,bsmote,70.000,12.857,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,86.439,bsmote,85.169,1.270,WIN
1,f1,casb_relaxed,63.049,smote,62.473,0.575,WIN
2,precision,casb_relaxed,55.090,smote,53.682,1.408,WIN
3,recall,casb_edited,84.643,adasyn,81.786,2.857,WIN


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.845154,0.833333,1.000000,0.714286,1.000000,0.857143,1.000000,1.000000,0.839244,None
1,42,1,svm,smote,0.50,0.958140,0.736842,0.583333,1.000000,0.918033,0.959016,0.863946,0.978923,0.847491,None
2,42,1,svm,bsmote,0.50,0.887066,0.666667,0.545455,0.857143,0.918033,0.887588,0.845971,0.976581,0.776866,None
3,42,1,svm,ros,0.75,0.862978,0.571429,0.428571,0.857143,0.868852,0.862998,0.798810,0.960187,0.717203,None
4,42,1,svm,adasyn,1.00,0.932123,0.636364,0.466667,1.000000,0.868852,0.934426,0.741017,0.957845,0.784243,None
5,42,1,svm,casb,0.75,0.932123,0.636364,0.466667,1.000000,0.868852,0.934426,0.818707,0.969555,0.784243,None
6,42,1,svm,casb_relaxed,1.00,0.940875,0.666667,0.500000,1.000000,0.885246,0.942623,0.869048,0.978923,0.803771,None
7,42,1,svm,casb_edited,0.50,0.940875,0.666667,0.500000,1.000000,0.885246,0.942623,0.937925,0.992974,0.803771,None
8,42,1,svm,casb_ros,0.75,0.923287,0.608696,0.437500,1.000000,0.852459,0.926230,0.814286,0.967213,0.765991,None
9,42,1,knn,original,0.50,0.925820,0.923077,1.000000,0.857143,1.000000,0.928571,0.982143,0.997658,0.924449,None
